# Data Cleaning & Data Analytics

## CPlus-Umfrage

Im Rahmen eines CPlus-Projektes wurden zwölf Cluster in sechs mitteleuropäischen Ländern untersucht. 

Dazu wurde ein Fragebogen erstellt, bei dem jede latente Variable anhand mehrerer Items operationalisiert wurde. Für jedes Item wurde eine Aussage formuliert, die mithilfe einer 7-Punkte-Likert-Skala („Stimme überhaupt nicht zu“, …, „Stimme voll und ganz zu“) beantwortet werden konnte.

Übersicht über die Items

* CD3-CD5 ... Daten zum Unternehmen (Gründungsjahr, Anzahl der Angestellten, Land)
* CL_NR ... Cluster-Nummer innerhalb eines Landes
* TR1-TR4 ... Statements zur Trust in Region (1 fully disagree, 7 fully agree)
* DI1-DI4 ...  Statements zu Environmental Dynamism (1 fully disagree, 7 fully agree)
* EC1-EC3 ... Statements zu Environmental Competitiveness (1 fully disagree, 7 fully agree)
* IP1-IP5 ... Statements zu Exploitative Innovation (1 fully disagree, 7 fully agree)
* IP6-IP14 ... Statements zu Explorative Innovation (1 fully disagree, 7 fully agree)
* PI1-PI3 ... Statements zu Process Innovation  (1 fully disagree, 7 fully agree)
* CI1-CI9 ... Statements zu Acquire and Assimilate Knowledge (1 fully disagree, 7 fully agree)
* TI1-TI12 ... Statements zu Transform and Exploit Knowledge (1 fully disagree, 7 fully agree)
* BS1-BS6 ... Statements zu Business Strategy (1 fully disagree, 7 fully agree)
* CM0-CM4  ... Statements zu Kooperation with Cluster Management (1 fully disagree, 7 fully agree)
* FP1-FP3  ... Statements zu Financial Performance (1 fully disagree, 7 fully agree)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
sns.set(style="whitegrid", context="notebook")

# Teil: Daten einlesen, erste Übersicht und Korrektur


In [ ]:
# Einlesen der Daten
df = pd.read_csv("daten/CPlus-Rohdaten.csv")
df.shape

In [ ]:
# Ausgabe der ersten Zeilen, Dimensionen und Spaltennamen
df.head()

In [ ]:
# einige statistische Kennzahlen
df.describe().round(2)

In [ ]:
# Anzahl an fehlenden Werten: keine?
df.isna().sum().sum()

## Fehlende Werte codieren

In [ ]:
# Für spezielle Spalten
df["CD4"] = df["CD4"].replace(-66, np.nan)
df["IP14"] = df["IP14"].replace(-77, np.nan)

In [ ]:
# Für Likert-Skalen (1..7)
likert_cols = [
    "TR1", "TR2", "TR3", "TR4", "DI1", "DI2", "DI3", "DI4", "EC1", "EC2", "EC3",
    "IP1", "IP2", "IP3", "IP4", "IP5", "IP6", "IP7", "IP8", "IP9", "IP10", "IP11",
    "PI1", "PI2", "PI3", "CI1", "CI2", "CI3", "CI4", "CI5", "CI6", "CI7", "CI8", "CI9",
    "TI1", "TI2", "TI3", "TI4", "TI5", "TI6", "TI7", "TI8", "TI9", "TI10", "TI11", "TI12",
    "BS1", "BS2", "BS3", "BS4", "BS5", "BS6", "CM1", "CM2", "CM3", "CM4",
    "FP1", "FP2", "FP3",
]

# 8 kann nicht vorkommen -> Codierung für NA
df[likert_cols] = df[likert_cols].replace(8, np.nan)

In [ ]:
# hier gibt es keine zusätzlichen Werte (= alle haben Frage beantwortet)
df["CM0"].value_counts()

In [ ]:
# Skala war nur von 1..5
df["IP13"].value_counts()

In [ ]:
# Deshalb ist 6 eine "Nicht-Antwort" -> NA
df["IP13"] = df["IP13"].replace(6, np.nan)

In [ ]:
df.isna().sum()

## Werte anpassen

In [ ]:
# Likert-Skala anpassen
# Negative Fragen einfach umkodieren: aus 7 wird 1 und umgekehrt.
reverse_cols = ["DI3", "CI4", "CI5", "CI7", "TI4", "TI5", "TI8", "TI11", "BS6", "CM2"]
df[reverse_cols] = df[reverse_cols].apply(lambda v: 8 - v)

In [ ]:
# Weitere Daten ableiten
# CD5=Land, CL_NR=1 oder 2 (es gab 2 Cluster pro Land)
df["ClusterNr"] = (df["CD5"] * 10 + df["CL_NR"]).astype("Int64").astype("category")
df["FirmAge"] = 2012 - df["CD3"]  # Umfrage fand 2012 statt
df[["ClusterNr", "FirmAge"]].head()

In [ ]:
# Und weil die Daten so schieflastig sind -> Log-Transformation
df["FirmAgeLog"] = np.log(df["FirmAge"])
df["FirmSizeLog"] = np.log(df["CD4"])

In [ ]:
# Nützliche Definitionen für später
items_all = list(df.columns)

# nur Variablen, d.h. keine Clusternummer, untransformierte Firmengröße, etc. IP14 siehe unten
items_vars = [c for c in items_all if c not in ["CD3", "CD4", "CD5", "CL_NR", "ClusterNr", "IP14", "FirmAge", "CM0"]]

# latent -> keine harten Fakten
items_latent = [c for c in items_vars if c not in ["FirmAgeLog", "FirmSizeLog"]]

# Abhängige Variable für Regressionsmodelle
items_abh = [
    "IP1", "IP2", "IP3", "IP4", "IP5",
    "IP6", "IP7", "IP8", "IP9", "IP10", "IP11",
    "PI1", "PI2", "PI3",
    "FP1", "FP2", "FP3",
    "IP13",
]

# alle anderen sind unabhängige Variable
items_unabh = [c for c in items_latent if c not in items_abh]

# Analyse der fehlenden Werte

In [ ]:
# Welche Spalten haben viele fehlende Werte?
df.isna().sum().sort_values(ascending=False).head(15)

In [ ]:
# Wie viele Datensätze sind komplett bzw. unvollständig?
n_records = len(df)
complete_records = df.dropna().shape[0]
n_records, complete_records, complete_records / n_records

# 100 Datensätze sind unvollständig. Da muss uns etwas einfallen, wir können nicht die Hälfte der Datensätze verwerfen.

In [ ]:
# Bei welchen Zeilen fehlt besonders viel Information?
df.isna().sum(axis=1).sort_values(ascending=False).head(10)

### Grafische Übersicht

In [ ]:
import missingno as mno

mno.bar(df)

In [ ]:
mno.matrix(df)

# Gut zu erkennen: es gibt Gruppen von Fragen (rechts) die immer gemeinsam nicht beantwortet werden

In [ ]:
mno.heatmap(df)

# hier sieht man die hohe Korrelation der Nicht-Beantwortung (rechts unten) sowie einzelne Zusammenhänge verstreut

In [ ]:
mno.dendrogram(df)

Das Dendrogramm liest man **von rechts nach links**.

Rechts stehen die einzelnen Spaltennamen, also zum Beispiel: FP1, FP2, FP3, IP14

Von rechts nach links sieht man dann, welche Spalten zuerst zusammengefasst werden. Je weiter rechts zwei Spalten verbunden werden, desto ähnlicher sind sie sich in Bezug auf fehlende Werte. Level 0 bedeutet typischerweise "kein Unterschied".

Eine Verbindung sehr weit rechts bedeutet also: Diese Spalten haben fast das gleiche Muster. Wenn in Spalte A ein Wert fehlt, fehlt er meistens auch in Spalte B. Wenn in Spalte A ein Wert vorhanden ist, ist er meistens auch in Spalte B vorhanden.

Beispiel: CI4 und TI1 werden bei 0 verbunden. Dann gilt:
* CI4 fehlt genau in denselben Zeilen wie TI1.
* TI1 ist genau in denselben Zeilen ausgefüllt wie CI4.

Wenn zwei Spalten nahe bei 0, aber nicht genau bei 0 verbunden werden, bedeutet das: Sie sind sich **sehr ähnlich**, aber nicht perfekt gleich.

Beispiel: FP1 und FP3 werden bei Abstand 1 verbunden. Dann heißt das:
* FP1 und FP3 fehlen meistens gemeinsam.
* FP1 und FP3 sind meistens gemeinsam ausgefüllt.
* Aber es gibt einige Zeilen, in denen sie sich unterscheiden.

Wenn eine Gruppe erst weiter links mit einer anderen Spalte verbunden wird, ist die Ähnlichkeit schwächer.

Beispiel: FP1 und FP3 werden zuerst zusammengefasst. Danach kommt FP2 dazu. Später kommt IP14 dazu.

Dann kann man sagen:
* FP1 und FP3 haben ein sehr ähnliches Muster fehlender Werte.
* FP2 ist diesen beiden Spalten auch ähnlich, aber nicht ganz so stark.
* IP14 gehört nur noch grob zu dieser Gruppe.

Das Dendrogramm hilft also dabei zu erkennen: 
Welche Spalten fehlen oft gemeinsam?
Welche Spalten gehören offenbar zum selben Themenblock?

In [ ]:
def misspair(x, y):
    x_miss = df[x].isna().astype(int)
    y_miss = df[y].isna().astype(int)
    corr = x_miss.corr(y_miss) if x_miss.std() > 1e-8 and y_miss.std() > 1e-8 else np.nan
    print(f"{x}=na: {x_miss.sum()}, {y}=na: {y_miss.sum()}, " # wie viele NA in jeder Spalte?
          f"{x}=na->{y}=na: {y_miss[x_miss==1].sum()}, "      # wenn Spalte X=na, wie viele Y=na?
          f"{y}=na->{x}=na: {x_miss[y_miss==1].sum()}, "      # wenn Spalte Y=na, wie viele X=na?
          f"miss_corr({x},{y})={corr:.2f}")

misspair("FP1", "FP3")  # sehr ähnlich
misspair("FP1", "FP2")  # etwas weniger ähnlich
misspair("FP1", "IP14") # noch weniger ähnlich
# etc.